In [1]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from PIL import Image, ImageSequence
import glob
import os
import re
import random
import astropy

from astropy import constants as const
from astropy import units as u

credit: @pmocz https://github.com/pmocz/nbody-python/blob/master/nbody.py

The first thing we need to do is define all the physics that is going on under the hood. The main thing we will be using for this is Newton's Law of Universal Gravitation (assuming no relativistic effects due to extreme speeds or densities). The net force on any particular body is the sum of the gravitational forces from all the other bodies pulling on it, so we can use this to calculate the accelerations, velocities, and positions as a function of time. This will allow us to then take a snapshot at a particular time, then time step and take another snapshot, repeating until the simulation ends. For very close approaches, where the gravitational force will increase to near infinity due to the very small separation distance, we will also implement a softening length that fixes the maximum gravitational force to be a constant.

In [159]:
# the first thing we need is to define a function that accepts positions, masses, and constants, and calculates the acceleration on each body.

def get_bary_accel(pos, mass, softening):

# pos is an N x 3 matrix of position (x,y,z) for each of the N bodies in the simulation at a particular time
# mass is an N x 1 vector of masses for each of the N bodies
# G and softening_length are Newton's gravity constant and the value used to "smooth out" the gravitational force, respectively

    # positions
    x = pos[:, 0:1]    # returns the first column of pos, the x values
    y = pos[:, 1:2]    # returns the second column of pos, the y values
    z = pos[:, 2:3]    # returns the third column of pos, the z values

    # pairwise particle separations
    dx = x.T - x
    dy = y.T - y
    dz = z.T - z

    # applying the inverse square law as vectors
    inv_r3 = (dx**2 +  dy**2 + dz**2 + softening**2)
    inv_r3[inv_r3 > 0] = inv_r3[inv_r3 > 0] ** (-1.5)

    # calculating accelerations: ma = GMm/r^2, so a = GM/r^2 
    a_x = (G * (dx * inv_r3) @ mass)
    a_y = (G * (dy * inv_r3) @ mass)
    a_z = (G *(dz * inv_r3) @ mass)

    # putting the components together
    a = np.hstack((a_x, a_y, a_z))

    # return the acceleration for use in other calculations
    return a

# now let's calculate KE and PE of the system
def get_energy(pos, vel, mass, G):

# pos is an N x 3 matrix of position (x,y,z) like before
# vel is an N x 3 matrix of velocities(v_x, v_y, v_z) for each body at a particular time
# mass is an N x 1 matrix of masses like before
# G is Newton's gravitational constant

    # calculate kinetic energy
    KE = 0.5 * np.sum(mass * vel**2)

    # positions
    x = pos[:, 0:1]
    y = pos[:, 1:2]
    z = pos[:, 2:3]

    # pairwise particle separations
    dx = x.T - x
    dy = y.T - y
    dz = z.T - z

    # calculate potential energy, U = -GMm/r
    inv_r = np.sqrt(dx**2 + dy**2 + dz**2)
    inv_r[inv_r > 0] = 1.0 / inv_r[inv_r > 0]

    # sum over upper triangle of the matrix to prevent double counting interactions
    PE = G * np.sum(np.triu(-(mass * mass.T) * inv_r, 1))

    return KE, PE

Now that the physics has been defined, let's input the actual parameters of the simulation and show how all the functions interact!

In [160]:
# taken from Google AI

def generate_disk_points(num_points, disk_radius):
    r_rand = np.random.rand(num_points) + 0.0025
    r = disk_radius * np.sqrt(r_rand)
    theta = 2 * np.pi * np.random.rand(num_points)

    firstx = r * np.cos(theta)
    firsty = r * np.sin(theta)

    finalx = np.append(firstx, 0.0)
    finaly = np.append(firsty, 0.0)
    finalz = np.zeros(len(firstx)+1)
    
    zipped_output = zip(finalx, finaly, finalz)
    result_list = list(zipped_output)
    xyz_list = np.array([list(triple) for triple in result_list])

    return xyz_list

In [194]:
def get_dm_accel(pos, softening):

# pos is an N x 3 matrix of position (x,y,z) for each of the N bodies in the simulation at a particular time
# mass is an N x 1 vector of masses for each of the N bodies

    # positions
    x = pos[:, 0:1]
    y = pos[:, 1:2]
    z = pos[:, 2:3]
    r = 0 * np.sqrt(pos[:, 0:1] **2 + pos[:, 1:2] **2 + pos[:, 2:3] **2)

    # pairwise particle separations
    dx = x.T - x
    dy = y.T - y
    dz = z.T - z

    #calculating DM content inside r
    Rvir = ((3 * tot_dm_mass) / (4*pi*scaled_virial_overdensity*crit_density))**(1.0/3)
    Rs = Rvir / c
    
    init_density = tot_dm_mass / (4*pi*(Rs**3)*(np.log(1+c)-(c/(1+c))))
    constant = (4*pi*(Rs**3)*init_density)

    dm_inside_r = constant * (np.log((Rs+r)/Rs) - r/(Rs+r)) * 0.25
    deriv_dm_inside_r = constant * (1/(Rs+r) * (r/(Rs+r)))

    # applying the inverse square law
    inv_r3 = (dx**2 +  dy**2 + dz**2 + softening**2)
    inv_r3[inv_r3 > 0] = inv_r3[inv_r3 > 0] ** (-1.5)

    inv_r = np.sqrt(dx**2 + dy**2 + dz**2)
    inv_r[inv_r > 0] = 1.0 / inv_r[inv_r > 0]
    
    # calculating accelerations:
    dm_a_x = (G * (dx * inv_r3) @ dm_inside_r) # (G * (dx * inv_r) @ deriv_dm_inside_r)
    dm_a_y = (G * (dy * inv_r3) @ dm_inside_r) # (G * (dy * inv_r) @ deriv_dm_inside_r)
    dm_a_z = (G * (dz * inv_r3) @ dm_inside_r) # (G * (dz * inv_r) @ deriv_dm_inside_r)

    # putting the components together
    dm_accel = np.hstack((dm_a_x, dm_a_y, dm_a_z))

    # return the acceleration for use in other calculations
    return (dm_accel, dm_inside_r)

In [209]:
# work out how to unify the random disk creation code with the rest of it (done)

def main():

    # time in years, masses in Msun, distances in pc if G = 1:
    
    # defining start time, end time, and timestep
    t = 0
    t_end = 0.25
    dt = 0.005
    plotRealTime = True

    # defining constants, # of bodies, softening length
    G = 1            # G = 1
    pi = np.pi
    c = 10
    scaled_virial_overdensity = 200
    crit_density = 9e-27 * (3.086e16)**3 / 2e30
    
    Msun = 1.989e30  # masses in kg
    pc = 3.086e16    # distances in parsecs
    
    plotlen = 1260    # box size in parsecs
    disk_radius = 2/3 * plotlen
    volume = (0.5 * plotlen) ** 3
    bodies = 999
    N_bodies = bodies + 1
    
    number_density = N_bodies / volume
    mean_interparticle_spacing = number_density ** (-1/3)
    softening_length = 0.1 * mean_interparticle_spacing

    # defining initial mass percentages
    tot_sim_mass = 1e7
    tot_dm_mass = 0.8 * tot_sim_mass
    tot_bary_mass = 0.2 * tot_sim_mass
    bh_mass = 0.999 * tot_bary_mass
    tot_stellar_mass = tot_bary_mass - bh_mass
    randmass = np.random.rand(bodies, 1)
    sum_randmass = np.sum(randmass)
    norm_stellar_masses = (randmass / sum_randmass) * tot_stellar_mass
    mass_array = np.append(norm_stellar_masses, bh_mass).reshape(-1, 1)
    mass = mass_array
    
    size = np.array(mass_array, dtype = np.float64)

    equiv_timestep1 = np.sqrt(((plotlen * 3.086e16) ** 3) / ((6.67e-11) * (tot_sim_mass * 2e30)))
    equiv_timestep2 = equiv_timestep1 / (60 * 60 * 24 * 365)

    pos_results = generate_disk_points(bodies, disk_radius)
    pos = pos_results

    x_pos = pos_results[:, 0]
    y_pos = pos_results[:, 1]
    z_pos = pos_results[:, 2]

    dx_pos = x_pos.T[:-1]
    dy_pos = y_pos.T[:-1]
    dz_pos = z_pos.T[:-1]

    inv_init_dist1 = 1.0 / (np.sqrt(dx_pos**2 + dy_pos**2 + dz_pos**2))
    inv_init_dist2 = np.append(inv_init_dist1, 0.0)
    inv_init_dist3 = inv_init_dist2.reshape(N_bodies, 1)

    v_x = -y_pos
    v_y = x_pos
    v_z = z_pos

    v_stack = np.column_stack((v_x, v_y, v_z))[:-1]
    magnitude_v = np.linalg.norm(v_stack, axis = 1)
    test = magnitude_v.reshape(bodies, 1)
    normalized_v = v_stack / test

    input_vel = np.sqrt((bh_mass + get_dm_accel(pos, softening)[1] + tot_stellar_mass) * inv_init_dist3)
    vel_coeff = 1    #(31536000 / 3.08e16)   # convert to parsec / yr
    test_vel = np.append(normalized_v, [[0.0, 0.0, 0.0]], axis = 0)
    vel = test_vel * input_vel * vel_coeff

    # print out initial conditions:
    print(f"Box Size: {plotlen} pc, Disk Radius: {disk_radius} pc")
    print(f"Total Simulated Mass: {tot_sim_mass:.2e} Msun")
    print(f"{bh_mass + np.sum(get_dm_accel(pos, softening)[1]) + tot_stellar_mass}")
    print(f"Dark Matter Mass: {tot_dm_mass:.2e} Msun")
    print(f"Baryonic Mass: {tot_bary_mass:.2e} Msun")
    print(f"Central SMBH Mass: {bh_mass:.2e} Msun")
    print(f"Total Stellar Mass: {tot_stellar_mass:.2e} Msun")
    print(f"Equivalent Timestep in years: {equiv_timestep2} yrs")
    print(f"# of Bodies: {N_bodies}, Softening Length: {softening_length} pc")
    
    
    # converting to COM frame
    vel -= (np.mean(mass * vel, 0) / np.mean(mass))

    # calculate initial gravitational accelerations
    acc = get_bary_accel(pos, mass, softening_length) + get_dm_accel(pos, softening_length)[0]
    
    # calculate initial KE and PE of the system
    KE, PE = get_energy(pos, vel, mass, G)

    # number of timesteps
    N_steps = int(np.ceil(t_end / dt))
    #N_steps_arr = np.arange(1, N_steps, 1)

    # save energies, particle orbits for plotting trails
    save_pos = np.zeros((N_bodies, 3, N_steps + 1))
    save_pos[:, :, 0] = pos
    save_KE = np.zeros(N_steps + 1)
    save_KE[0] = KE
    save_PE = np.zeros(N_steps + 1)
    save_PE[0] = PE
    t_all = np.arange(N_steps + 1) * dt

    
    # prep figure
    fig = plt.figure(constrained_layout = True)
    axs = fig.subplot_mosaic([['main', 'right']], gridspec_kw={'width_ratios':[2, 1]})
    axs['right'].set_title('Rotation Curve')
    #axs['bottomright'].set_title('KE And PE At Each Timestep')

    
### now let's run the simulation!!
    
    for i in range(N_steps):      # compute kinematics:
        
        # 1/2 step kick
        vel += (acc * dt)/2.0
        
        # full-step drift
        pos += (vel * dt)

        ## update accelerations:
        acc = get_bary_accel(pos, mass, softening_length) + get_dm_accel(pos, softening_length)[0]
        vel += (acc * dt)/2.0

        ## update time:
        t += dt

        # get the KE and PE of the system:
        KE, PE = get_energy(pos, vel, mass, G)

        # save the KE and PE for plotting:
        save_pos[:, :, i + 1] = pos
        save_KE[i + 1] = KE
        save_PE[i + 1] = PE

        if plotRealTime or (i == N_steps - 1):
            # make the n-body sim plot
            plt.sca(axs['main'])
            plt.cla()

            xx = save_pos[:, 0, max(i - 50, 0) : i + 1]
            yy = save_pos[:, 1, max(i - 50, 0): i + 1]

            xx_last = save_pos[-1, 0, max(i - 50, 0) : i + 1]
            yy_last = save_pos[-1, 1, max(i - 50, 0): i + 1]

            plot_pos_x = pos[:, 0][:-1]
            plot_pos_y = pos[:, 1][:-1]

            plot_pos_central_x = pos[:, 0][-1]
            plot_pos_central_y = pos[:, 1][-1]

            #axs[0].scatter(xx, yy, s = 5, color = 'tab:blue', alpha = 0.33)
            axs['main'].scatter(plot_pos_x, plot_pos_y, s = size[:-1], color = 'tab:blue')
                
            #axs[0].scatter(xx_last, yy_last, s = 5, color = 'black', alpha = 0.33)
            axs['main'].scatter(plot_pos_central_x, plot_pos_central_y, s = 0.000001 * mass_array[-1], color = 'black')

            axs['main'].set_xlim(-plotlen, plotlen)
            axs['main'].set_ylim(-plotlen, plotlen)

            axs['main'].text(-plotlen + 0.75, 0.9 * plotlen, f"KE: {KE:.2e}", fontsize = 6, color = "tab:red")
            axs['main'].text(-plotlen + 0.75, 0.8 * plotlen, f"PE: {PE:.2e}", fontsize = 6, color = "tab:blue")
            axs['main'].text(-plotlen + 0.75, 0.7 * plotlen, f"KE/PE Ratio: {(KE / PE):.2f}", fontsize = 6, color = "tab:purple")
            axs['main'].text(-plotlen + 0.75, 0.6 * plotlen, f"Total Energy: {(KE + PE):.2e}", fontsize = 6, color = "black")
            axs['main'].set_title(f"Frame {i}", fontsize = 10)
            plt.suptitle(f"N-body Simulation of {N_bodies} Particles")

            # make a rotation curve for the n-body sim
            plt.sca(axs['right'])
            #plt.cla()

            v_vect = vel[:, :-1]
            r_vect = np.column_stack((pos[:, 0], pos[:, 1]))
            r_vect_T = r_vect.T
            v_dot_r = np.diag(np.dot(v_vect, r_vect_T))
            mag_r = np.sqrt(np.diag(np.dot(r_vect, r_vect_T)))
            scalar_factor = v_dot_r / mag_r
            rad_vel = scalar_factor[:, np.newaxis] * r_vect
            tan_vel = v_vect - rad_vel
            mag_tan_vel = np.sqrt((tan_vel[:, 0])**2 + (tan_vel[:, 1])**2)

            rot_curve_vals1 = list(zip(mag_r, mag_tan_vel))
            rot_curve_vals2 = sorted(rot_curve_vals1, key=lambda x: x[0])
            radii, tan_vels = zip(*rot_curve_vals2)

            axs['right'].scatter(radii[1:], tan_vels[1:], s = 2.5, color = 'tab:green')
            axs['right'].set_xlabel("Distance from Center")
            axs['right'].set_ylabel("Tangential Velocity")
            #axs['right'].set_yscale("log")

            # save figure
            plt.savefig(f'images/{i:004}', dpi = 150)
            plt.close()

        
if __name__== "__main__":
  main()

Box Size: 1260 pc, Disk Radius: 840.0 pc
Total Simulated Mass: 1.00e+07 Msun
384436619.81028545
Dark Matter Mass: 8.00e+06 Msun
Baryonic Mass: 2.00e+06 Msun
Central SMBH Mass: 2.00e+06 Msun
Total Stellar Mass: 2.00e+03 Msun
Equivalent Timestep in years: 210506339.10489956 yrs
# of Bodies: 1000, Softening Length: 6.299999999999999 pc


In [210]:
#current_dir = os.getcwd()
#print(current_dir)

# Define the path to the file containing all the PNG images
image_directory = "C:/Users/steph/1 DM TUTORIAL/images/*.png"
output_gif_path = "new_nbodysim.gif"

image_paths = sorted(glob.glob(image_directory), key=lambda x: int(re.search(r"(\d+)", x).group()))
 
# Load images 
images = []
for path in image_paths:
    img = Image.open(path)
    images.append(img)

if images:
    images[0].save(output_gif_path, save_all=True, append_images=images[1:], duration = 75, loop=0)
    print(f"GIF created and saved at {output_gif_path}")
else:
    print("No images found in the specified directory.")

GIF created and saved at new_nbodysim.gif
